In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum,when,udf, monotonically_increasing_id, sum as spark_sum
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time
from pyspark.sql.types import DoubleType
from functools import reduce



In [2]:
#Creating a spark session
spark = SparkSession.builder \
    .appName("AppendixCancerRiskPrediction") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 13:41:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#Loading the dataset and creating a spark dataframe
df = spark.read.csv(
    "appendix_cancer_prediction_dataset.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- Patient_ID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoking_Status: string (nullable = true)
 |-- Alcohol_Consumption: string (nullable = true)
 |-- Family_History_Cancer: string (nullable = true)
 |-- Genetic_Mutations: string (nullable = true)
 |-- Chronic_Diseases: string (nullable = true)
 |-- Physical_Activity_Level: string (nullable = true)
 |-- Diet_Type: string (nullable = true)
 |-- Radiation_Exposure: string (nullable = true)
 |-- Previous_Cancers: string (nullable = true)
 |-- Blood_Pressure: integer (nullable = true)
 |-- Cholesterol_Level: integer (nullable = true)
 |-- White_Blood_Cell_Count: double (nullable = true)
 |-- Red_Blood_Cell_Count: double (nullable = true)
 |-- Platelet_Count: integer (nullable = true)
 |-- Tumor_Markers: string (nullable = true)
 |-- Symptom_Severity: string (nullable = true)
 |-- Diagnosis_Delay_

### Data cleaning

In [4]:
#Removing the duplicate records
df = df.dropDuplicates()
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

#Since there are no NULL values, there is no need of handling null values


26/05/03 13:41:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:=============================>                             (1 + 1) / 2]

+----------+-------+---+------+---+--------------+-------------------+---------------------+-----------------+----------------+-----------------------+---------+------------------+----------------+--------------+-----------------+----------------------+--------------------+--------------+-------------+----------------+--------------------+--------------+------------------------------+--------------------------+
|Patient_ID|Country|Age|Gender|BMI|Smoking_Status|Alcohol_Consumption|Family_History_Cancer|Genetic_Mutations|Chronic_Diseases|Physical_Activity_Level|Diet_Type|Radiation_Exposure|Previous_Cancers|Blood_Pressure|Cholesterol_Level|White_Blood_Cell_Count|Red_Blood_Cell_Count|Platelet_Count|Tumor_Markers|Symptom_Severity|Diagnosis_Delay_Days|Treatment_Type|Survival_Years_After_Diagnosis|Appendix_Cancer_Prediction|
+----------+-------+---+------+---+--------------+-------------------+---------------------+-----------------+----------------+-----------------------+---------+---------

#### There is a clear class balance with label=0 with count=220713 and label=1 and count=39287

In [5]:
target_col = "Appendix_Cancer_Prediction"

# df_model = df.withColumn(
#     "label",
#     when(col(target_col) == "Yes", 1.0)
#     .when(col(target_col) == "No", 0.0)
# ).drop("Patient_ID", target_col)

label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label"
)
df = label_indexer.fit(df).transform(df)
df_model = df.drop("Patient_ID", target_col)
df_model.groupBy("label").count().show()
df.select(target_col, "label").distinct().show()

+-----+------+
|label| count|
+-----+------+
|  0.0|220713|
|  1.0| 39287|
+-----+------+



[Stage 23:=============================>                            (1 + 1) / 2]

+--------------------------+-----+
|Appendix_Cancer_Prediction|label|
+--------------------------+-----+
|                        No|  0.0|
|                       Yes|  1.0|
+--------------------------+-----+



### Train-test split

In [6]:
total = df_model.count()
minority_count = df_model.filter(col("label") == 1).count()
majority_count = df_model.filter(col("label") == 0).count()

weight_for_0 = total / (2 * majority_count)
weight_for_1 = total / (2 * minority_count)

df_model = df_model.withColumn(
    "weight",
    when(col("label") == 1, weight_for_1).otherwise(weight_for_0)
)



#extracting columns which have labels/categories as values

leakage_cols = [
    "Survival_Years_After_Diagnosis",
    "Diagnosis_Delay_Days",
    "Treatment_Type",
    "Tumor_Markers"
]

df_model = df_model.drop(*leakage_cols)

categorical_cols = [
    field.name for field in df_model.schema.fields
    if field.dataType.simpleString() == "string"
]

numeric_cols = [
    field.name for field in df_model.schema.fields
    if field.dataType.simpleString() != "string" and field.name not in ["label", "weight"]
    
]


# print("Categorical:", categorical_cols)
# print("Numeric:", numeric_cols)

indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep")
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=col + "_index", outputCol=col + "_encoded")
    for col in categorical_cols
]

feature_cols = numeric_cols + [col + "_encoded" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
indexed_categorical_cols = [c + "_index" for c in categorical_cols]
gbt_feature_cols = numeric_cols + indexed_categorical_cols

gbt_assembler = VectorAssembler(
    inputCols=gbt_feature_cols,
    outputCol="features"
)



train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

train_df = train_df.cache()
test_df = test_df.cache()
train_df.count()
test_df.count()

51840

### Logistic Regression

In [7]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=200,
    regParam=0.001,
    elasticNetParam=0.0,
    threshold=0.5
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler,  lr])
time_start = time.time()
lr_model = lr_pipeline.fit(train_df)
time_end = time.time()
time_taken = time_end - time_start
print(f"Time taken to train logistic regression is {time_taken}")


lr_predictions = lr_model.transform(test_df)

26/05/03 13:45:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
                                                                                

Time taken to train logistic regression is 135.86493635177612


In [8]:
lr_predictions.select("label", "prediction", "probability").show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |1.0       |[0.49185648870135595,0.5081435112986441]|
|0.0  |1.0       |[0.49300220604764844,0.5069977939523516]|
|0.0  |1.0       |[0.4865453450648355,0.5134546549351645] |
|1.0  |1.0       |[0.4786154714205545,0.5213845285794455] |
|0.0  |1.0       |[0.4790953752265146,0.5209046247734854] |
|0.0  |1.0       |[0.4778747074482346,0.5221252925517654] |
|0.0  |1.0       |[0.4787012704322063,0.5212987295677938] |
|0.0  |1.0       |[0.47850313222210533,0.5214968677778946]|
|0.0  |1.0       |[0.4898320111935402,0.5101679888064599] |
|0.0  |1.0       |[0.49000617216131204,0.509993827838688] |
+-----+----------+----------------------------------------+
only showing top 10 rows



In [9]:
lr_predictions.groupBy("label", "prediction").count().show()

[Stage 157:====================================================>(198 + 2) / 200]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 3736|
|  0.0|       1.0|21260|
|  1.0|       0.0| 4108|
|  0.0|       0.0|22736|
+-----+----------+-----+



In [10]:
lr_cm = lr_predictions.select(
    spark_sum(when((col("label") == 1.0) & (col("prediction") == 1.0), 1).otherwise(0)).alias("TP"),
    spark_sum(when((col("label") == 0.0) & (col("prediction") == 1.0), 1).otherwise(0)).alias("FP"),
    spark_sum(when((col("label") == 1.0) & (col("prediction") == 0.0), 1).otherwise(0)).alias("FN"),
    spark_sum(when((col("label") == 0.0) & (col("prediction") == 0.0), 1).otherwise(0)).alias("TN")
)

lr_cm.show()

[Stage 162:===================================================> (195 + 2) / 200]

+----+-----+----+-----+
|  TP|   FP|  FN|   TN|
+----+-----+----+-----+
|3736|21260|4108|22736|
+----+-----+----+-----+



In [11]:
lr_metrics = lr_cm.withColumn(
    "Positive Precision",
    col("TP") / (col("TP") + col("FP"))
).withColumn(
    "Positive Recall",
    col("TP") / (col("TP") + col("FN"))
)

lr_metrics.show()

[Stage 167:===================================================> (196 + 2) / 200]

+----+-----+----+-----+------------------+-------------------+
|  TP|   FP|  FN|   TN|Positive Precision|    Positive Recall|
+----+-----+----+-----+------------------+-------------------+
|3736|21260|4108|22736|0.1494639142262762|0.47628760836308004|
+----+-----+----+-----+------------------+-------------------+



### Random Forest

In [13]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    maxDepth=10,
    maxBins=64,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler,  rf])

rf_start_time = time.time()
rf_model = rf_pipeline.fit(train_df)
rf_end_time = time.time()

rf_training_time = rf_end_time - rf_start_time
print(f"Time taken to run Random Forest is = {rf_training_time}")

rf_predictions = rf_model.transform(test_df)

26/05/03 14:06:50 WARN DAGScheduler: Broadcasting large task binary with size 1194.6 KiB
26/05/03 14:07:44 WARN DAGScheduler: Broadcasting large task binary with size 1985.8 KiB
26/05/03 14:09:25 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/05/03 14:11:47 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB
26/05/03 14:14:14 WARN DAGScheduler: Broadcasting large task binary with size 1040.0 KiB
26/05/03 14:15:02 WARN DAGScheduler: Broadcasting large task binary with size 8.7 MiB
26/05/03 14:18:25 WARN DAGScheduler: Broadcasting large task binary with size 1552.4 KiB
                                                                                

Time taken to run Random Forest is = 975.1785917282104


In [14]:
rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

26/05/03 14:19:29 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB


+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0.0  |0.0       |[0.5208483877617647,0.4791516122382353] |
|0.0  |0.0       |[0.5222737066330823,0.47772629336691774]|
|0.0  |0.0       |[0.5048563604518236,0.4951436395481764] |
|1.0  |0.0       |[0.5239226188248475,0.47607738117515247]|
|0.0  |0.0       |[0.5093493147310272,0.49065068526897293]|
|0.0  |1.0       |[0.4941731583525381,0.5058268416474619] |
|0.0  |0.0       |[0.5108613359298139,0.48913866407018625]|
|0.0  |1.0       |[0.49972648900070993,0.5002735109992901]|
|0.0  |0.0       |[0.5044657043862961,0.495534295613704]  |
|0.0  |0.0       |[0.5365134710945916,0.4634865289054083] |
|1.0  |1.0       |[0.49017001675189575,0.5098299832481041]|
|0.0  |1.0       |[0.4984990073028219,0.5015009926971782] |
|0.0  |0.0       |[0.5139247240180983,0.48607527598190164]|
|0.0  |0.0       |[0.5187635725461702,0.

In [15]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

In [16]:

print("Logistic Regression AUC:", auc_evaluator.evaluate(lr_predictions))
print("Logistic Regression Accuracy:", accuracy_evaluator.evaluate(lr_predictions))
print("Logistic Regression F1:", f1_evaluator.evaluate(lr_predictions))
print("Random Forest Precision:", precision_evaluator.evaluate(lr_predictions))
print("Random Forest Recall:", recall_evaluator.evaluate(lr_predictions))


print("Random Forest AUC:", auc_evaluator.evaluate(rf_predictions))
print("Random Forest Accuracy:", accuracy_evaluator.evaluate(rf_predictions))
print("Random Forest F1:", f1_evaluator.evaluate(rf_predictions))
print("Random Forest Precision:", precision_evaluator.evaluate(rf_predictions))
print("Random Forest Recall:", recall_evaluator.evaluate(rf_predictions))


Logistic Regression AUC: 0.49694463236169223


Logistic Regression Accuracy: 0.5106481481481482


Logistic Regression F1: 0.5791982183928645


Random Forest Precision: 0.7414271669317851


Random Forest Recall: 0.5106481481481482


...

Random Forest F1: 0.6974589796983826


26/05/03 14:26:04 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
                                                                                

Random Forest Precision: 0.7439310981001069


26/05/03 14:27:37 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
[Stage 420:==========================================>          (159 + 2) / 200]

Random Forest Recall: 0.6631365740740741


In [17]:
# lr_predictions.groupBy("label", "prediction").count().show()
rf_predictions.groupBy("label", "prediction").count().show()

26/05/03 14:29:10 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/05/03 14:30:46 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
                                                                                

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 2120|
|  0.0|       1.0|11739|
|  1.0|       0.0| 5724|
|  0.0|       0.0|32257|
+-----+----------+-----+



### GBT Classifier

In [18]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=50,
    maxDepth=4,
    stepSize=0.05,
    maxBins=64,
    seed=42
)

gbt_pipeline = Pipeline(stages=indexers + [gbt_assembler, gbt])

gbt_start_time = time.time()
gbt_model = gbt_pipeline.fit(train_df)
gbt_end_time = time.time()

gbt_training_time = gbt_end_time - gbt_start_time
gbt_predictions = gbt_model.transform(test_df)
print(f"Training time for GBT classifier {gbt_training_time}")
print("GBT AUC:", auc_evaluator.evaluate(gbt_predictions))
print("GBT Accuracy:", accuracy_evaluator.evaluate(gbt_predictions))
print("GBT F1:", f1_evaluator.evaluate(gbt_predictions))
print("GBT Precision:", precision_evaluator.evaluate(gbt_predictions))
print("GBT Recall:", recall_evaluator.evaluate(gbt_predictions))



gbt_predictions.groupBy("label", "prediction").count().show()

Training time for GBT classifier 1138.9829654693604


GBT AUC: 0.500412066341945


GBT Accuracy: 0.5341820987654321


GBT F1: 0.6001139399408926


GBT Precision: 0.7443201968774577


GBT Recall: 0.5341820987654321


[Stage 1121:===================================================>(198 + 2) / 200]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 3583|
|  0.0|       1.0|19887|
|  1.0|       0.0| 4261|
|  0.0|       0.0|24109|
+-----+----------+-----+



### Trying to increase the positive class weight

In [19]:
from pyspark.sql.functions import col, when

# Count each class in the training set
class_counts = train_df.groupBy("label").count().collect()

count_dict = {row["label"]: row["count"] for row in class_counts}

negative_count = count_dict[0.0]
positive_count = count_dict[1.0]

# Positive class weight = number of negative samples / number of positive samples
positive_weight = negative_count / positive_count

print("Negative count:", negative_count)
print("Positive count:", positive_count)
print("Positive class weight:", positive_weight)

# Add weight column
train_df_weighted = train_df.withColumn(
    "weight",
    when(col("label") == 1.0, positive_weight).otherwise(1.0)
)

# Optional: check weights
train_df_weighted.groupBy("label", "weight").count().show()

Negative count: 176717
Positive count: 31443
Positive class weight: 5.620233438285151


[Stage 1131:==================================================> (194 + 2) / 200]

+-----+-----------------+------+
|label|           weight| count|
+-----+-----------------+------+
|  1.0|5.620233438285151| 31443|
|  0.0|              1.0|176717|
+-----+-----------------+------+



In [20]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    maxDepth=10,
    maxBins=64,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])
start_time = time.time()
rf_model_positive_weighted = rf_pipeline.fit(train_df_weighted)
end_time = time.time()
rf_predictions_positive_weighted = rf_model_positive_weighted.transform(test_df)
time_taken = end_time - start_time
print(f"Time taken is {time_taken}") 



26/05/03 14:55:01 WARN DAGScheduler: Broadcasting large task binary with size 1195.2 KiB
26/05/03 14:56:02 WARN DAGScheduler: Broadcasting large task binary with size 1986.3 KiB
26/05/03 14:57:42 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/05/03 15:00:13 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB
26/05/03 15:02:51 WARN DAGScheduler: Broadcasting large task binary with size 1038.4 KiB
26/05/03 15:03:39 WARN DAGScheduler: Broadcasting large task binary with size 8.7 MiB
26/05/03 15:07:16 WARN DAGScheduler: Broadcasting large task binary with size 1551.8 KiB
                                                                                

Time taken is 1028.5426080226898


In [21]:
print("RF AUC:", auc_evaluator.evaluate(rf_predictions_positive_weighted))
print("RF Accuracy:", accuracy_evaluator.evaluate(rf_predictions_positive_weighted))
print("RF F1:", f1_evaluator.evaluate(rf_predictions_positive_weighted))
print("RF Precision:", precision_evaluator.evaluate(rf_predictions_positive_weighted))
print("RF Recall:", recall_evaluator.evaluate(rf_predictions_positive_weighted))


26/05/03 15:08:21 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
                                                                                

RF AUC: 0.5022113149083741


26/05/03 15:10:22 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
                                                                                

RF Accuracy: 0.6603395061728395


26/05/03 15:11:57 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
                                                                                

RF F1: 0.6955522084963707


26/05/03 15:13:34 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
                                                                                

RF Precision: 0.7435952167878663


26/05/03 15:15:09 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
[Stage 1257:===================================================>(198 + 2) / 200]

RF Recall: 0.6603395061728394


In [22]:
rf_predictions_positive_weighted.groupBy("label", "prediction").count().show()

26/05/03 15:16:44 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/05/03 15:18:20 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
                                                                                

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 2137|
|  0.0|       1.0|11901|
|  1.0|       0.0| 5707|
|  0.0|       0.0|32095|
+-----+----------+-----+



In [23]:
pwrf_cm = rf_predictions_positive_weighted.select(
    spark_sum(when((col("label") == 1.0) & (col("prediction") == 1.0), 1).otherwise(0)).alias("TP"),
    spark_sum(when((col("label") == 0.0) & (col("prediction") == 1.0), 1).otherwise(0)).alias("FP"),
    spark_sum(when((col("label") == 1.0) & (col("prediction") == 0.0), 1).otherwise(0)).alias("FN"),
    spark_sum(when((col("label") == 0.0) & (col("prediction") == 0.0), 1).otherwise(0)).alias("TN")
)

pwrf_cm.show()

26/05/03 15:18:22 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
[Stage 1265:===================================================>(199 + 1) / 200]

+----+-----+----+-----+
|  TP|   FP|  FN|   TN|
+----+-----+----+-----+
|2137|11901|5707|32095|
+----+-----+----+-----+



In [24]:
pwrf_metrics = pwrf_cm.withColumn(
    "Positive Precision",
    col("TP") / (col("TP") + col("FP"))
).withColumn(
    "Positive Recall",
    col("TP") / (col("TP") + col("FN"))
)

pwrf_metrics.show()

26/05/03 15:19:58 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
[Stage 1270:===================================================>(198 + 2) / 200]

+----+-----+----+-----+------------------+------------------+
|  TP|   FP|  FN|   TN|Positive Precision|   Positive Recall|
+----+-----+----+-----+------------------+------------------+
|2137|11901|5707|32095|0.1522296623450634|0.2724375318714941|
+----+-----+----+-----+------------------+------------------+



### Save the models

In [25]:
lr_model.write().overwrite().save("saved_models/logistic_regression_model")

In [26]:
rf_model_positive_weighted.write().overwrite().save("saved_models/rf_model_positive_weighted")

26/05/03 15:22:05 WARN TaskSetManager: Stage 1478 contains a task of very large size (3331 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [27]:
rf_model.write().overwrite().save("saved_models/random_forest_model")

26/05/03 15:22:19 WARN TaskSetManager: Stage 1583 contains a task of very large size (3343 KiB). The maximum recommended task size is 1000 KiB.


In [28]:
gbt_model.write().overwrite().save("saved_models/gbt_model")